# 09 Seaborn Visualizations (Advanced)

Advanced Seaborn visual narratives for sessions and ML pipeline outputs from DuckDB.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from utils.duckdb_utils import query_df, table_exists
from utils.paths import get_db_paths, resolve_workspace

sns.set_theme(style='ticks', context='talk')

workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

sessions = query_df(input_db, 'SELECT title, day, track, description FROM sessions_in_preprocessed')
sessions['desc_len'] = sessions['description'].fillna('').str.len()
            


In [ ]:
plt.figure(figsize=(12, 6))
sns.ecdfplot(data=sessions, x='desc_len', hue='day')
plt.title('Description Length ECDF by Day')
plt.tight_layout()
plt.show()

In [ ]:
if output_db.exists() and table_exists(output_db, 'clustering_assignments'):
    cl = query_df(output_db, 'SELECT track, CAST(cluster_id AS VARCHAR) AS cluster_id FROM clustering_assignments')
    top_tracks = cl['track'].value_counts().head(10).index
    heat = cl[cl['track'].isin(top_tracks)].pivot_table(index='track', columns='cluster_id', aggfunc='size', fill_value=0)
    plt.figure(figsize=(11, 7))
    sns.heatmap(heat, cmap='crest', annot=True, fmt='d')
    plt.title('Track to Cluster Archetypes')
    plt.tight_layout()
    plt.show()
else:
    print('Run notebook 04 first.')
            


In [ ]:
if output_db.exists() and table_exists(output_db, 'model_selection_results'):
    ms = query_df(output_db, 'SELECT * FROM model_selection_results')
    plt.figure(figsize=(9, 5))
    sns.pointplot(data=ms, x='model', y='cv_f1_mean', join=False, color='darkorange')
    plt.title('Model Selection CV Comparison')
    plt.tight_layout()
    plt.show()
else:
    print('Run notebook 06 first.')
            


In [ ]:
if output_db.exists() and table_exists(output_db, 'ml_artifacts'):
    art = query_df(output_db, 'SELECT notebook, metrics_json FROM ml_artifacts')
    rows = []
    for _, r in art.iterrows():
        m = json.loads(r['metrics_json']) if r['metrics_json'] else {}
        for k in ['weighted_f1', 'holdout_f1_weighted', 'silhouette', 'explained_total']:
            if k in m:
                rows.append({'notebook': r['notebook'], 'metric': k, 'value': float(m[k])})
    proc = pd.DataFrame(rows)
    if not proc.empty:
        plt.figure(figsize=(11, 5))
        sns.barplot(data=proc, x='notebook', y='value', hue='metric')
        plt.xticks(rotation=25, ha='right')
        plt.title('ML Pipeline Metrics from Artifact Metadata')
        plt.tight_layout()
        plt.show()
            
